# TDE-HMM — Master Report

In [ ]:
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, yaml
from pathlib import Path
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 120})

PROJECT_NAME = "eeg_hmm_plattform"
current = Path.cwd()
while current.name != PROJECT_NAME:
    if current.parent == current:
        raise RuntimeError(f"No se encontro {PROJECT_NAME}")
    current = current.parent
PROJECT_ROOT = current

# Experimentos TDE a comparar
TDE_EXPERIMENTS = [
    "canonical/tde_k3_t7.yaml",
    "canonical/tde_k4_t7.yaml",
    "canonical/tde_k5_t7.yaml",
    "canonical/tde_k4_t15.yaml",
    "canonical/tde_k3_t15.yaml",
    "canonical/tde_k5_t15.yaml",
]

STATE_COLORS = ["#E63946","#2196F3","#4CAF50","#FF9800","#9C27B0","#00BCD4"]
FIG_DIR = PROJECT_ROOT / "reports" / "figures" / "tde_master"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def save_fig(name):
    plt.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

print(f"proyecto: {PROJECT_ROOT}")
print(f"experimentos configurados: {len(TDE_EXPERIMENTS)}")


---
## Carga de experimentos

In [ ]:
def load_experiment(yaml_rel):
    yaml_path = PROJECT_ROOT / "configs" / "experiments" / yaml_rel
    if not yaml_path.exists():
        return None
    with open(yaml_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    exp_name   = cfg["experiment"]["name"]
    K          = cfg["pipeline"]["hmm"]["k_states"]
    n_lags     = cfg["tde"]["n_lags"]
    cov_type   = cfg["pipeline"]["hmm"]["covariance_type"]
    pca_var    = cfg["pipeline"]["pca"]["variance_retained"]

    output_dir = PROJECT_ROOT / cfg["paths"]["output_dir"].replace("../../","").replace("../","")
    exp_dir    = output_dir / exp_name

    required = [
        exp_dir / f"hmm_model_k{K}.pkl",
        exp_dir / f"viterbi_paths_k{K}.npy",
        exp_dir / "decoding_summary.json",
        exp_dir / "state_stats.csv",
        exp_dir / "fo_by_subject.csv",
    ]
    missing = [r.name for r in required if not r.exists()]
    if missing:
        print(f"  SKIP {exp_name}: faltan {missing}")
        return None

    summary  = json.loads((exp_dir / "decoding_summary.json").read_text())
    df_stats = pd.read_csv(exp_dir / "state_stats.csv")
    df_fo    = pd.read_csv(exp_dir / "fo_by_subject.csv")
    manifest = json.loads((exp_dir / "feature_manifest.json").read_text())

    sfreq   = cfg["tde"].get("sfreq", 250.0)
    step_ms = 1000.0 / sfreq

    print(f"  OK {exp_name} | K={K} | t={n_lags} | ",
          f"score={summary[chr(115)+chr(99)+chr(111)+chr(114)+chr(101)+chr(95)+chr(52)]}/4 | ",
          f"sessions={summary[chr(110)+chr(95)+chr(115)+chr(101)+chr(115)+chr(115)+chr(105)+chr(111)+chr(110)+chr(115)]}")

    return {
        "name":     exp_name,
        "K":        K,
        "n_lags":   n_lags,
        "cov_type": cov_type,
        "pca_var":  pca_var,
        "step_ms":  step_ms,
        "exp_dir":  exp_dir,
        "summary":  summary,
        "df_stats": df_stats,
        "df_fo":    df_fo,
        "manifest": manifest,
        "label":    f"K={K} t={n_lags}",
        "score":    summary[chr(115)+chr(99)+chr(111)+chr(114)+chr(101)+chr(95)+chr(52)],
        "verdict":  summary[chr(118)+chr(101)+chr(114)+chr(100)+chr(105)+chr(99)+chr(116)],
    }

print("Cargando experimentos TDE...")
experiments = []
for yaml_rel in TDE_EXPERIMENTS:
    exp = load_experiment(yaml_rel)
    if exp:
        experiments.append(exp)

print(f"
{len(experiments)}/{len(TDE_EXPERIMENTS)} experimentos cargados")


---
## Tabla comparativa

In [ ]:
rows = []
for e in experiments:
    s = e["summary"]
    rows.append({
        "Experimento": e["name"],
        "K":           e["K"],
        "n_lags":      e["n_lags"],
        "D_raw":       e["manifest"].get("D_raw", "-"),
        "n_PCs":       e["manifest"].get("n_pcs", "-"),
        "FO_min":      s["FO_min"],
        "FO_max":      s["FO_max"],
        "dwell_max_ms":s["dwell_max_ms"],
        "confidence":  s["confidence"],
        "n_absorbing": s["n_absorbing"],
        "Score":       s[chr(115)+chr(99)+chr(111)+chr(114)+chr(101)+chr(95)+chr(52)],
        "Veredicto":   s[chr(118)+chr(101)+chr(114)+chr(100)+chr(105)+chr(99)+chr(116)],
    })

df_cmp = pd.DataFrame(rows)
display(Markdown("### Comparativa de experimentos TDE"))
display(df_cmp.style
    .background_gradient(subset=["FO_min"], cmap="RdYlGn")
    .background_gradient(subset=["dwell_max_ms"], cmap="RdYlGn_r")
    .background_gradient(subset=["confidence"], cmap="Greens")
    .background_gradient(subset=["Score"], cmap="RdYlGn")
    .format({"FO_min":"{:.4f}","FO_max":"{:.4f}",
             "dwell_max_ms":"{:.1f}","confidence":"{:.4f}"})
)


---
## FO por estado — comparativa K y lags

In [ ]:
if experiments:
    t7_exps  = [e for e in experiments if e["n_lags"]==7]
    t15_exps = [e for e in experiments if e["n_lags"]==15]

    def plot_fo_sweep(exps, title):
        if not exps: return
        fig, axes = plt.subplots(1, len(exps), figsize=(4*len(exps), 4.5), sharey=True)
        if len(exps) == 1: axes = [axes]
        for ax, e in zip(axes, sorted(exps, key=lambda x: x["K"])):
            stats_e = e["df_stats"]
            fo_vals = stats_e["FO_hard"].values
            colors  = [STATE_COLORS[s] for s in range(e["K"])]
            ax.bar([f"S{s}" for s in range(e["K"])], fo_vals, color=colors, alpha=0.85)
            ax.axhline(1/e["K"], color="gray", linestyle="--", alpha=0.5)
            for s, fo in enumerate(fo_vals):
                ax.text(s, fo+0.005, f"{fo:.3f}", ha="center", fontsize=9)
            ax.set_title(f"K={e[chr(75)]}  score={e[chr(115)+chr(99)+chr(111)+chr(114)+chr(101)]}/4",
                         fontweight="bold")
            ax.set_ylabel("FO" if exps.index(e)==0 else "")
            ax.set_ylim(0, max(fo_vals)*1.2+0.05)
        plt.suptitle(title, fontsize=13, fontweight="bold")
        plt.tight_layout()
        save_fig(f"fo_sweep_{title[:3].lower()}")
        plt.show()

    plot_fo_sweep(t7_exps,  "t=7 lags: K sweep")
    plot_fo_sweep(t15_exps, "t=15 lags: K sweep")


---
## Dwell times — comparativa

In [ ]:
if experiments:
    rows_dw = []
    for e in experiments:
        for _, row in e["df_stats"].iterrows():
            rows_dw.append({
                "experimento": e["name"],
                "K":          e["K"],
                "n_lags":     e["n_lags"],
                "estado":     row["state"],
                "dwell_analitico": row["dwell_analytic_ms"],
                "dwell_mediana":   row["dwell_median_ms"],
                "self_trans":      row["self_trans"],
            })
    df_dw = pd.DataFrame(rows_dw)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, col, title in zip(axes,
        ["dwell_analitico", "dwell_mediana"],
        ["Dwell analitico (ms)", "Dwell mediana (ms)"]):
        for e in experiments:
            sub = df_dw[df_dw["experimento"]==e["name"]]
            ax.plot(sub["estado"], sub[col], marker="o",
                    label=e["label"], lw=2, alpha=0.85)
        ax.axhline(2000, color="red", linestyle="--", alpha=0.5, label="limite 2s")
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel("Estado"); ax.set_ylabel("ms")
        ax.legend(fontsize=8)
    plt.suptitle("Dwell times — comparativa TDE", fontsize=13, fontweight="bold")
    plt.tight_layout(); save_fig("dwell_comparison"); plt.show()


---
## Absorbentes por experimento

In [ ]:
if experiments:
    rows_abs = []
    for e in experiments:
        rows_abs.append({
            "Experimento": e["name"],
            "K":           e["K"],
            "n_lags":      e["n_lags"],
            "n_absorbing": e["summary"]["n_absorbing"],
            "pct_absorbing": round(100*e["summary"]["n_absorbing"]/e["summary"]["n_sessions"],1),
        })
    df_abs = pd.DataFrame(rows_abs)
    display(df_abs.style
        .background_gradient(subset=["pct_absorbing"], cmap="Reds")
        .format({"pct_absorbing":"{:.1f}%"})
    )


---
## Veredicto final

In [ ]:
print("="*60)
print("VEREDICTO FINAL — TDE-HMM")
print("="*60)

for e in sorted(experiments, key=lambda x: (-x["score"], x["K"], x["n_lags"])):
    v = e["verdict"]
    tag = "REPORTAR" if e["score"]==4 else "ACEPTABLE" if e["score"]==3 else "REVISAR"
    print(f"  {tag:<10} {e[chr(110)+chr(97)+chr(109)+chr(101)]:<35} score={e[chr(115)+chr(99)+chr(111)+chr(114)+chr(101)]}/4")

print()
best = [e for e in experiments if e["score"]==4]
if best:
    print(f"Aptos para reporte: {len(best)}")
    for e in best:
        print(f"  {e[chr(110)+chr(97)+chr(109)+chr(101)]}")
else:
    print("Ningun experimento paso score 4/4 todavia (esperado — pendientes HPC)")
